# Seleção de modelos com scikit-learn

Este notebook integra preparação, estimators, pipelines, validação cruzada e busca de hiperparâmetros. Ele cobre regressão, classificação e clusterização, mantendo o teste final fora de todas as decisões supervisionadas. Consulte o [guia do módulo](../docs/004__MODEL_SELECTION.md) para a fundamentação.

## Estimators e `Pipeline`

Um estimator aprende em `fit()`. Transformers, como imputer, scaler e PCA, também expõem `transform()`; o estimator final, como `Ridge` ou `SVC`, prevê em `predict()`. Dentro de uma `Pipeline`, cada transformer é ajustado apenas com a parte de treino de cada dobra, evitando data leakage.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import loguniform, t
from sklearn.cluster import DBSCAN, KMeans
from sklearn.datasets import load_breast_cancer, load_diabetes, load_wine
from sklearn.decomposition import PCA
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import (adjusted_rand_score, mean_absolute_error, mean_squared_error,
                             r2_score, silhouette_score)
from sklearn.model_selection import (GridSearchCV, KFold, RandomizedSearchCV,
                                     StratifiedKFold, cross_validate, train_test_split)
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

RANDOM_STATE = 42
pd.set_option('display.float_format', lambda value: f'{value:.3f}')

In [3]:
pipeline_exemplo = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', Ridge()),
])

# Os nomes das etapas compõem os hiperparâmetros pesquisáveis.
print(pipeline_exemplo.get_params().keys() >= {'model__alpha'})
pipeline_exemplo

True


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto accou

## 1. Regressão: `KFold` e `GridSearchCV`

Usamos `load_diabetes`. Primeiro, reservamos 20% como teste final. As comparações e a busca ocorrem apenas no conjunto de desenvolvimento. `KFold` é adequado porque a tarefa é regressão e as observações do dataset são tratadas como independentes.

In [4]:
diabetes = load_diabetes(as_frame=True)
X_reg, y_reg = diabetes.data, diabetes.target
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=RANDOM_STATE
)

cv_reg = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
modelos_reg = {
    'baseline': DummyRegressor(strategy='mean'),
    'ridge': make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), Ridge()),
    'random forest': RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE),
}

comparacao_reg = []
for nome, modelo in modelos_reg.items():
    scores = cross_validate(modelo, X_reg_train, y_reg_train, cv=cv_reg,
                            scoring={'rmse': 'neg_root_mean_squared_error', 'r2': 'r2'})
    comparacao_reg.append({
        'modelo': nome, 'RMSE médio': -scores['test_rmse'].mean(),
        'RMSE desvio': scores['test_rmse'].std(), 'R² médio': scores['test_r2'].mean(),
    })
display(pd.DataFrame(comparacao_reg).sort_values('RMSE médio'))

,modelo,RMSE médio,RMSE desvio,R² médio
1,ridge,55.374,2.373,0.481
2,random forest,58.895,1.915,0.410
0,baseline,78.134,5.638,-0.027


`GridSearchCV` avalia todas as combinações de uma grade. Aqui a grade é pequena e interpretável: é uma boa situação para busca exaustiva. O prefixo `model__` referencia a última etapa do pipeline.

In [5]:
ridge_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', Ridge()),
])
param_grid = {'model__alpha': [0.01, 0.1, 1.0, 10.0, 100.0]}
grid_reg = GridSearchCV(
    ridge_pipeline, param_grid=param_grid, scoring='neg_root_mean_squared_error',
    cv=cv_reg, refit=True, return_train_score=True
)
grid_reg.fit(X_reg_train, y_reg_train)

print('Melhores parâmetros:', grid_reg.best_params_)
print(f'Melhor RMSE de CV: {-grid_reg.best_score_:.3f}')
display(pd.DataFrame(grid_reg.cv_results_)[
    ['param_model__alpha', 'mean_test_score', 'std_test_score', 'rank_test_score']
].sort_values('rank_test_score'))

Melhores parâmetros: {'model__alpha': 1.0}
Melhor RMSE de CV: 55.374


,param_model__alpha,mean_test_score,std_test_score,rank_test_score
2,1.000,-55.374,2.373,1
1,0.100,-55.389,2.364,2
0,0.010,-55.394,2.362,3
3,10.000,-55.435,2.355,4
4,100.000,-56.001,2.853,5


In [6]:
# O teste é consultado somente depois de congelar a escolha do pipeline.
pred_reg = grid_reg.best_estimator_.predict(X_reg_test)
erros_abs = np.abs(y_reg_test - pred_reg)
rmse_test = mean_squared_error(y_reg_test, pred_reg) ** 0.5
mae_test = mean_absolute_error(y_reg_test, pred_reg)
intervalo_mae = t.interval(0.95, len(erros_abs) - 1, loc=erros_abs.mean(), scale=erros_abs.sem())
print(f'RMSE final: {rmse_test:.3f} | MAE final: {mae_test:.3f} | R² final: {r2_score(y_reg_test, pred_reg):.3f}')
print(f'IC 95% aproximado para erro absoluto médio: ({intervalo_mae[0]:.3f}, {intervalo_mae[1]:.3f})')

RMSE final: 53.777 | MAE final: 42.812 | R² final: 0.454
IC 95% aproximado para erro absoluto médio: (35.918, 49.706)


## 2. Classificação: `StratifiedKFold`, Random Search, Bayesiana e Optuna

Em classificação, `StratifiedKFold` preserva a proporção de classes. `C` e `gamma` de uma SVC variam em muitas ordens de grandeza; `RandomizedSearchCV` usa distribuições logarítmicas e um orçamento fixo de candidatos, em vez de enumerar uma grade enorme.

In [7]:
cancer = load_breast_cancer(as_frame=True)
X_clf, y_clf = cancer.data, cancer.target
X_clf_train, X_clf_test, y_clf_train, y_clf_test = train_test_split(
    X_clf, y_clf, test_size=0.20, random_state=RANDOM_STATE, stratify=y_clf
)
cv_clf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
svc_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', SVC(probability=True, random_state=RANDOM_STATE)),
])

random_search = RandomizedSearchCV(
    svc_pipeline,
    param_distributions={
        'model__C': loguniform(1e-2, 1e3),
        'model__gamma': loguniform(1e-4, 1e0),
        'model__kernel': ['rbf'],
    },
    n_iter=20, scoring='roc_auc', cv=cv_clf, random_state=RANDOM_STATE, refit=True
)
random_search.fit(X_clf_train, y_clf_train)
print('Melhores parâmetros:', random_search.best_params_)
print(f'Melhor ROC-AUC de CV: {random_search.best_score_:.3f}')
display(pd.DataFrame(random_search.cv_results_)[
    ['param_model__C', 'param_model__gamma', 'mean_test_score', 'std_test_score', 'rank_test_score']
].sort_values('rank_test_score').head())

/home/marqs/dev/scikit-learn-class/venv/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/marqs/dev/scikit-learn-class/venv/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/marqs/dev/scikit-learn-class/venv/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/marqs/dev/scikit-learn-class/venv/lib/python3.13/site-packages/sklearn/svm/_base.py:239: Fu

Melhores parâmetros: {'model__C': np.float64(26.373339933815235), 'model__gamma': np.float64(0.005762487216478602), 'model__kernel': 'rbf'}
Melhor ROC-AUC de CV: 0.996


/home/marqs/dev/scikit-learn-class/venv/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/marqs/dev/scikit-learn-class/venv/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/marqs/dev/scikit-learn-class/venv/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/marqs/dev/scikit-learn-class/venv/lib/python3.13/site-packages/sklearn/svm/_base.py:239: Fu

,param_model__C,param_model__gamma,mean_test_score,std_test_score,rank_test_score
19,26.373,0.006,0.996,0.006,1
6,145.282,0.001,0.995,0.006,2
15,10.907,0.000,0.994,0.005,3
10,11.462,0.000,0.994,0.005,3
8,0.332,0.013,0.994,0.005,5


In [8]:
pred_clf = random_search.best_estimator_.predict(X_clf_test)
proba_clf = random_search.best_estimator_.predict_proba(X_clf_test)[:, 1]
from sklearn.metrics import f1_score, roc_auc_score
print(f'ROC-AUC final: {roc_auc_score(y_clf_test, proba_clf):.3f}')
print(f'F1 final (limiar padrão): {f1_score(y_clf_test, pred_clf):.3f}')

ROC-AUC final: 0.996
F1 final (limiar padrão): 0.986


### Busca Bayesiana e Optuna — opções avançadas

As células seguintes são opcionais. Elas usam o mesmo pipeline, métrica e CV da busca aleatória. Bayesiana usa o histórico dos trials para propor candidatos; Optuna permite definir a função-objetivo diretamente. Ambas devem ser comparadas sob orçamento semelhante, não apenas pelo melhor escore encontrado.

In [9]:
try:
    from skopt import BayesSearchCV
    from skopt.space import Real
    HAS_SKOPT = True
except ImportError:
    HAS_SKOPT = False

if HAS_SKOPT:
    bayes_search = BayesSearchCV(
        svc_pipeline,
        search_spaces={'model__C': Real(1e-2, 1e3, prior='log-uniform'),
                       'model__gamma': Real(1e-4, 1e0, prior='log-uniform')},
        n_iter=20, scoring='roc_auc', cv=cv_clf, random_state=RANDOM_STATE, refit=True
    )
    bayes_search.fit(X_clf_train, y_clf_train)
    print('BayesSearchCV:', bayes_search.best_params_, f'ROC-AUC={bayes_search.best_score_:.3f}')
else:
    print('Seção opcional: instale scikit-optimize para executar BayesSearchCV.')

/home/marqs/dev/scikit-learn-class/venv/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/marqs/dev/scikit-learn-class/venv/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/marqs/dev/scikit-learn-class/venv/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/marqs/dev/scikit-learn-class/venv/lib/python3.13/site-packages/sklearn/svm/_base.py:239: Fu

BayesSearchCV: OrderedDict({'model__C': 115.33999859559563, 'model__gamma': 0.0004869521028037488}) ROC-AUC=0.996


/home/marqs/dev/scikit-learn-class/venv/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [10]:
try:
    import optuna
    from sklearn.model_selection import cross_val_score
    HAS_OPTUNA = True
except ImportError:
    HAS_OPTUNA = False

if HAS_OPTUNA:
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    def objective(trial):
        pipeline = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
            ('model', SVC(
                C=trial.suggest_float('C', 1e-2, 1e3, log=True),
                gamma=trial.suggest_float('gamma', 1e-4, 1e0, log=True),
                probability=True, random_state=RANDOM_STATE,
            )),
        ])
        return cross_val_score(pipeline, X_clf_train, y_clf_train, cv=cv_clf, scoring='roc_auc').mean()

    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
    study.optimize(objective, n_trials=20)
    print('Optuna:', study.best_params, f'ROC-AUC={study.best_value:.3f}')
else:
    print('Seção opcional: instale optuna para executar a busca por trials.')

/home/marqs/dev/scikit-learn-class/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/marqs/dev/scikit-learn-class/venv/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/marqs/dev/scikit-learn-class/venv/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/marqs/dev/scikit-learn-class/venv/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probabili

Optuna: {'C': 21.653305905841258, 'gamma': 0.004018340594759094} ROC-AUC=0.996


/home/marqs/dev/scikit-learn-class/venv/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/marqs/dev/scikit-learn-class/venv/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/marqs/dev/scikit-learn-class/venv/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/marqs/dev/scikit-learn-class/venv/lib/python3.13/site-packages/sklearn/svm/_base.py:239: Fu

## 3. Clusterização: seleção de configuração

Clusterização não possui um alvo para CV supervisionada. Não usaremos `GridSearchCV` como se K-Means ou DBSCAN fossem classificadores. Em vez disso, compararemos silhouette, tamanho dos grupos e estabilidade sob inicializações; em DBSCAN, a proporção de ruído também é obrigatória.

In [11]:
wine = load_wine(as_frame=True)
X_cluster = wine.data
X_cluster_scaled = make_pipeline(SimpleImputer(strategy='median'), StandardScaler()).fit_transform(X_cluster)

resultados_cluster = []
for representacao, dados in [
    ('escalonada', X_cluster_scaled),
    ('PCA 95%', PCA(n_components=0.95, random_state=RANDOM_STATE).fit_transform(X_cluster_scaled)),
]:
    for k in range(2, 7):
        labels = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE).fit_predict(dados)
        resultados_cluster.append({
            'representação': representacao, 'k': k, 'dimensões': dados.shape[1],
            'silhouette': silhouette_score(dados, labels), 'tamanhos': np.bincount(labels).tolist(),
        })

resultados_cluster = pd.DataFrame(resultados_cluster)
display(resultados_cluster.sort_values('silhouette', ascending=False))

,representação,k,dimensões,silhouette,tamanhos
6,PCA 95%,3,10,0.299,"[65, 51, 62]"
1,escalonada,3,13,0.285,"[65, 51, 62]"
7,PCA 95%,4,10,0.273,"[27, 49, 47, 55]"
5,PCA 95%,2,10,0.270,"[87, 91]"
0,escalonada,2,13,0.259,"[87, 91]"
2,escalonada,4,13,0.259,"[55, 49, 45, 29]"
8,PCA 95%,5,10,0.247,"[36, 48, 55, 32, 7]"
4,escalonada,6,13,0.237,"[48, 53, 28, 5, 19, 25]"
3,escalonada,5,13,0.232,"[26, 55, 48, 26, 23]"
9,PCA 95%,6,10,0.205,"[20, 27, 24, 31, 54, 22]"


In [12]:
# Estabilidade: ARI ignora a permutação arbitrária dos identificadores de cluster.
labels_por_semente = []
for seed in range(10):
    labels_por_semente.append(KMeans(n_clusters=3, n_init=1, random_state=seed).fit_predict(X_cluster_scaled))
aris = [adjusted_rand_score(labels_por_semente[0], labels) for labels in labels_por_semente[1:]]
print(f'Estabilidade contra a primeira inicialização: média ARI={np.mean(aris):.3f}, mínimo={np.min(aris):.3f}')

resultados_dbscan = []
for eps in [0.8, 1.0, 1.2, 1.4]:
    for min_samples in [4, 6, 8]:
        labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(X_cluster_scaled)
        mask = labels != -1
        n_clusters = len(set(labels[mask]))
        silhouette = silhouette_score(X_cluster_scaled[mask], labels[mask]) if n_clusters >= 2 else np.nan
        resultados_dbscan.append({
            'eps': eps, 'min_samples': min_samples, 'clusters': n_clusters,
            'ruído': 1 - mask.mean(), 'silhouette sem ruído': silhouette,
        })
display(pd.DataFrame(resultados_dbscan).sort_values('silhouette sem ruído', ascending=False))

Estabilidade contra a primeira inicialização: média ARI=0.969, mínimo=0.882


,eps,min_samples,clusters,ruído,silhouette sem ruído
0,0.800,4,0,1.000,NaN
1,0.800,6,0,1.000,NaN
2,0.800,8,0,1.000,NaN
3,1.000,4,0,1.000,NaN
4,1.000,6,0,1.000,NaN
5,1.000,8,0,1.000,NaN
6,1.200,4,0,1.000,NaN
7,1.200,6,0,1.000,NaN
8,1.200,8,0,1.000,NaN
9,1.400,4,0,1.000,NaN


Uma busca Bayesiana ou Optuna pode otimizar clusterização apenas depois que você definir uma função-objetivo responsável: ela deve rejeitar um único cluster, penalizar excesso de ruído e considerar estabilidade. Em exploração não supervisionada, a interpretação dos perfis nas features originais continua sendo o critério decisivo.

## Conclusões

- Pipelines tornam a preparação parte correta da validação, sem leakage.
- Escolha o splitter pela estrutura dos dados: `KFold` não serve para grupos ou séries temporais; classificação pede estratificação.
- Grid Search é didático para grades pequenas; Random Search, Bayesiana e Optuna usam melhor orçamentos em espaços amplos.
- O teste final é uma medição, não uma ferramenta de ajuste.
- Clusterização exige seleção por métricas internas, estabilidade e interpretação, não uma CV supervisionada automática.